In [3]:
import os, io, re, glob, zipfile
import numpy as np

PROJ  = os.path.expanduser("~/projects-ML-2026/project-1")
DATA  = os.path.join(PROJ, "data")
CACHE = os.path.join(DATA, "cache")
os.makedirs(CACHE, exist_ok=True)
print("data :", DATA)

data : /home/projects-ML-2026/project-1/data


## Unzipping cell:

In [4]:
def csv_dir():
    """Folder holding x_train.csv, unzipping the archive on first run."""
    hits = glob.glob(os.path.join(DATA, "**", "x_train.csv"), recursive=True)
    if not hits:
        zips = glob.glob(os.path.join(DATA, "*.zip"))
        if not zips:
            raise FileNotFoundError(f"no CSV and no zip in {DATA}")
        print("unzipping", os.path.basename(zips[0]), "...")
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(os.path.join(DATA, "unzipped"))
        hits = glob.glob(os.path.join(DATA, "**", "x_train.csv"), recursive=True)
    return os.path.dirname(hits[0])

SRC = csv_dir()
print("CSV folder :", SRC)
print(sorted(os.listdir(SRC)))

unzipping 81fdc2af-6811-4089-9054-9d59f417f819_dataset.zip ...
CSV folder : /home/projects-ML-2026/project-1/data/unzipped/dataset
['sample_submission.csv', 'x_test.csv', 'x_train.csv', 'y_train.csv']


In [5]:
def read_csv(path):
    """Parse a CSV into float32; empty cells become NaN. Returns (header, array)."""
    with open(path) as f:
        header = f.readline().strip().split(",")
        txt = f.read()
    txt = re.sub(r"(?<=,)(?=,|\n)", "nan", txt)      # cellule vide -> nan
    arr = np.loadtxt(io.StringIO(txt), delimiter=",")
    return header, arr.astype(np.float32)

hx, xtr = read_csv(os.path.join(SRC, "x_train.csv"))
hy, ytr = read_csv(os.path.join(SRC, "y_train.csv"))
ht, xte = read_csv(os.path.join(SRC, "x_test.csv"))

names  = hx[1:]
X_raw  = xtr[:, 1:]
XT_raw = xte[:, 1:]
ids_tr = xtr[:, 0].astype(np.int64)
ids_te = xte[:, 0].astype(np.int64)
y      = ytr[:, 1].astype(np.int8)
del xtr, xte, ytr
print("train", X_raw.shape, "| test", XT_raw.shape, "| y", y.shape)

train (328135, 321) | test (109379, 321) | y (328135,)


In [8]:
np.save(os.path.join(CACHE, "X_raw.npy"), X_raw)
np.save(os.path.join(CACHE, "XT_raw.npy"), XT_raw)
np.savez(os.path.join(CACHE, "meta.npz"), y=y, ids_tr=ids_tr, ids_te=ids_te,
         names=np.array(names))
for f in sorted(os.listdir(CACHE)):
    print(f"  {f:<16} {os.path.getsize(os.path.join(CACHE, f))/1e6:>8.0f} Mo")

  XT_raw.npy            140 Mo
  X_raw.npy             421 Mo
  meta.npz                4 Mo
